##Read & Write en Delta Lake

In [0]:

%sql
--Creamos la base de datos de Movie_demo
CREATE SCHEMA IF NOT EXISTS movie_demo
MANAGED LOCATION "abfss://demo@lsdata01.dfs.core.windows.net/

"

In [0]:
%run "../includes/librerias"

In [0]:
import pyspark.sql.functions

movie_schema = StructType ( fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )

#Leemos un archivo y cremoa el dataframe movie_df
movie_df = spark.read\
                .option("header", True)\
                .schema(movie_schema) \
                .csv("abfss://bronze@lsdata01.dfs.core.windows.net/2024-12-30/movie.csv")

In [0]:
#1. Escribir datos en Delta Lake (Managed Table)
# Utilizamos el DataFrame movie_df y Creamos una tabla administrada en formato delta
movie_df.write.format("delta").mode("overwrite").saveAsTable("movie_demo.movies_managed")

#2. Escribir datos en Delta Lake (External Table)
# O podemos Guarda los datos del dataframe movie_df en un directorio en formato delta
movie_df.write.format("delta").mode("overwrite").save("abfss://demo@lsdata01.dfs.core.windows.net/movies_external")

In [0]:
%sql
--3. con los datos que se guardaron en la carpeta external, podemos Crear una tabla utilizando los ficheros delta
create table movie_demo.movies_external
using delta
location "abfss://demo@lsdata01.dfs.core.windows.net/movies_external"

####3. Leer datos en Delta Lake (carpeta)

In [0]:
#Cargamos los datos de la carpeta delta en un dataframe
movies_external_df = spark.read.format("delta").load("abfss://demo@lsdata01.dfs.core.windows.net/movies_external")

####3. Leer datos en Delta Lake (Table)

In [0]:
#Creamos una tabla administrada por una particion
movie_df.write.format("delta").mode("overwrite").partitionBy("yearReleaseDate").saveAsTable("movie_demo.movies_partitioned")


In [0]:
%sql
describe extended movie_demo.movies_partitioned

#### Update desde Delta Lake

In [0]:
%sql
SELECT * 
FROM movie_demo.movies_managed 




In [0]:
%sql
-- Actualiza registros utilizando SQL

UPDATE movie_demo.movies_managed
SET durationTime = 60
WHERE yearReleaseDate = 2012


In [0]:
%sql
select durationTime from movie_demo.movies_managed where yearReleaseDate = 2012

In [0]:
##Actualiza registros utilizando SPython
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "movie_demo.movies_managed")
deltaTable.update(
                    condition = "yearReleaseDate = 2013",
                    set = {"durationTime": "100"}
                )

In [0]:
%sql
select durationTime from movie_demo.movies_managed where yearReleaseDate = 2013

In [0]:
%sql
-- Borra registros utilizando SQL

delete from movie_demo.movies_managed
WHERE yearReleaseDate = 2014;

select * from movie_demo.movies_managed
WHERE yearReleaseDate = 2014;


In [0]:
##Borra registros utilizando Python
from delta.tables import *

deltaTable = DeltaTable.forName(spark, "movie_demo.movies_managed")
deltaTable.delete(
                    "yearReleaseDate = 2015"
                 )

In [0]:
%sql
select * from movie_demo.movies_managed
WHERE yearReleaseDate = 2015;